# EconSight Phase 2 — Canadian Macroeconomic Analysis
VAR/VECM, XGBoost, SHAP, Monte Carlo, and Composite Economic Health Score.

In [ ]:
import asyncio
import warnings
warnings.filterwarnings("ignore")

import nest_asyncio
nest_asyncio.apply()

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

from econsight.db.connection import db_connection
from econsight.models.features import load_mart, build_feature_matrix
from econsight.models.var_model import VARModel
from econsight.models.xgb_model import XGBForecastModel, TARGETS, HORIZONS
from econsight.models.shap_analysis import compute_shap_summary
from econsight.models.monte_carlo import simulate
from econsight.models.composite import CompositeScorer

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100

async def _load():
    async with db_connection() as conn:
        return await load_mart(conn)

df_raw = asyncio.run(_load())
X = build_feature_matrix(df_raw)
print(f"Loaded {len(df_raw)} months  ({df_raw.index[0]} -> {df_raw.index[-1]})")
print(f"Feature matrix: {X.shape[1]} columns, {len(X)} rows")


## 1. Data Overview

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 9))
for i, col in enumerate(df_raw.columns):
    axes.ravel()[i].plot(df_raw.index, df_raw[col])
    axes.ravel()[i].set_title(col.replace("_", " ").title())
    axes.ravel()[i].tick_params(axis="x", rotation=45)
plt.suptitle("Canadian Macro Indicators -- Full History", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## 2. Feature Correlation & Stationarity

In [ ]:
from statsmodels.tsa.stattools import adfuller

print("ADF Stationarity Test (p < 0.05 -> stationary)")
for col in df_raw.columns:
    p = adfuller(df_raw[col].dropna())[1]
    status = "pass" if p < 0.05 else "fail"
    print(f"  [{status}] {col:<22} p={p:.4f}")


## 3. VAR/VECM Results

In [ ]:
var = VARModel()
var.fit(df_raw)
var_forecasts = var.predict(horizons=[1, 3])

print(f"Model type: {var._model_type}")
print(f"\n{'Target':<22} {'1-month':>10} {'3-month':>10}")
print("-" * 44)
for t in TARGETS:
    print(f"  {t:<20} {var_forecasts[1][t]:>10.3f} {var_forecasts[3][t]:>10.3f}")


## 4. XGBoost Results

In [ ]:
xgb_models = {}
rows = []
for target in TARGETS:
    for h in HORIZONS:
        y = df_raw[target].shift(-h).dropna()
        common = X.index.intersection(y.index)
        model = XGBForecastModel(target=target, horizon=h)
        metrics = model.fit(X.loc[common], y.loc[common])
        xgb_models[(target, h)] = model
        rows.append({"target": target, "horizon": h,
                     "train_rmse": round(metrics.train_rmse, 4),
                     "test_rmse": round(metrics.test_rmse, 4)})

print(pd.DataFrame(rows).to_string(index=False))


## 5. SHAP Analysis

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, target in enumerate(TARGETS):
    model = xgb_models[(target, 1)]
    summary = compute_shap_summary(model, X)
    top = summary.top_features[:10]
    vals = [summary.mean_abs[f] for f in top]
    axes[i].barh(top[::-1], vals[::-1])
    axes[i].set_title(f"SHAP -- {target} (h=1)")
    axes[i].set_xlabel("Mean |SHAP value|")
plt.suptitle("Feature Importance by SHAP (XGBoost)", fontsize=13)
plt.tight_layout()
plt.show()


## 6. Monte Carlo Uncertainty

In [ ]:
sim = simulate(xgb_models, X, n_sims=1000)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, target in enumerate(TARGETS):
    ax = axes[i]
    labels, p10s, p50s, p90s = [], [], [], []
    for h in HORIZONS:
        band = sim.bands[(target, h)]
        labels.append(f"h={h}")
        p10s.append(band["p10"]); p50s.append(band["p50"]); p90s.append(band["p90"])
    x = list(range(len(labels)))
    ax.bar(x, p50s, yerr=[
        [p50s[j] - p10s[j] for j in x],
        [p90s[j] - p50s[j] for j in x]
    ], capsize=6, alpha=0.7)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_title(target.replace("_", " ").title())
plt.suptitle("Monte Carlo: p10/p50/p90 Bands", fontsize=13)
plt.tight_layout()
plt.show()

print("\nNamed Scenarios (3-month horizon):")
print(f"{'Scenario':<12} {'CPI':>10} {'Unemployment':>14} {'Overnight':>12}")
print("-" * 50)
for name, vals in sim.scenarios.items():
    print(f"  {name:<10} {vals['cpi']:>10.3f} {vals['unemployment_rate']:>14.3f} {vals['overnight_rate']:>12.3f}")


## 7. Economic Health Score

In [ ]:
scorer = CompositeScorer()
scorer.fit(df_raw)
scores = scorer.score(df_raw)

plt.figure(figsize=(13, 4))
plt.fill_between(scores.index, scores["score"], alpha=0.25, color="seagreen")
plt.plot(scores.index, scores["score"], color="seagreen", linewidth=1.5)
plt.axhline(50, color="gray", linestyle="--", linewidth=0.8, label="Neutral (50)")
plt.title("Economic Health Score -- Canada (0=worst, 100=best)")
plt.ylabel("Score"); plt.legend(); plt.tight_layout(); plt.show()

latest = scores.iloc[-1]
print(f"Latest ({scores.index[-1]}): score = {latest['score']:.1f}/100")
cs = latest["component_scores"]
print("\nTop 5 contributors:")
for k, v in sorted(cs.items(), key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {k:<22}: {v:+.4f}")
